In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import ResNet50
import numpy as np

# 1. Setup Data
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 2. Define Model Architectures

def get_lenet():
    return models.Sequential([
        layers.Conv2D(6, (5, 5), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D(),
        layers.Conv2D(16, (5, 5), activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(120, activation='relu'),
        layers.Dense(84, activation='relu'),
        layers.Dense(10, activation='softmax')
    ], name="LeNet-5")

def get_alexnet():
    # Modified for 32x32 input
    return models.Sequential([
        layers.Conv2D(48, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(192, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(192, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(1024, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1024, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name="AlexNet_Modified")

def get_vgg16():
    # Standard VGG16 blocks
    model = models.Sequential(name="VGG-16")
    cfg = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M'] # Reduced depth for CIFAR
    model.add(layers.Input(shape=(32, 32, 3)))
    for v in cfg:
        if v == 'M':
            model.add(layers.MaxPooling2D((2, 2)))
        else:
            model.add(layers.Conv2D(v, (3, 3), padding='same', activation='relu'))
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dense(10, activation='softmax'))
    return model

def get_resnet50():
    # Using Keras built-in ResNet50
    base_resnet = ResNet50(include_top=False, weights=None, input_shape=(32, 32, 3))
    model = models.Sequential([
        base_resnet,
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation='softmax')
    ], name="ResNet-50")
    return model

# 3. Training Loop
model_factories = [get_lenet, get_alexnet, get_vgg16, get_resnet50]
results = {}

for factory in model_factories:
    model = factory()
    print(f"\n--- Training {model.name} ---")

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Training for 5 epochs only for demonstration (Increase to 50+ for full accuracy)
    model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=1)

    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results[model.name] = acc
    print(f"{model.name} Test Accuracy: {acc:.4f}")

# 4. Final Comparison
print("\n" + "="*30)
print("FINAL ACCURACY COMPARISON")
print("="*30)
for name, acc in results.items():
    print(f"{name:20}: {acc*100:.2f}%")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Training LeNet-5 ---
Epoch 1/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 32s 85ms/step - accuracy: 0.3561 - loss: 1.7541 - val_accuracy: 0.4414 - val_loss: 1.5210
Epoch 2/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 30s 84ms/step - accuracy: 0.4663 - loss: 1.4776 - val_accuracy: 0.4900 - val_loss: 1.3892
Epoch 3/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 30s 86ms/step - accuracy: 0.5079 - loss: 1.3711 - val_accuracy: 0.5134 - val_loss: 1.3472
Epoch 4/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 42s 88ms/step - accuracy: 0.5354 - loss: 1.2978 - val_accuracy: 0.5442 - val_loss: 1.2699
Epoch 5/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 29s 84ms/step - accuracy: 0.5572 - loss: 1.2434 - val_accuracy: 0.5516 - val_loss: 1.2430
LeNet-5 Test Accuracy: 0.5551

--- Training AlexNet_Modified ---
Epoch 1/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 512s 1s/step - accuracy: 0.3596 - loss: 1.6969 - val_accuracy: 0.5106 - val_loss: 1.3135
Epoch 2/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 571s 1s/step - accuracy: 0.5643 - loss: 1.2077 - val_accuracy: 0.5936 - val_loss: 1.1075
Epoch 3/

KeyboardInterrupt: 